# A7. Render the full attack set on Kaggle

Renders all 200 photos (50 tune + 150 test from `data/attack_set.csv`), applies the defences, and writes `full.csv`. Everything lands in `/kaggle/working`, which becomes the dataset the full runs read.

**What gets made, per photo (179 rows in the manifest)**
- Rendering sweep, no defence: the default `font16_high_object` plus 6 settings that change one thing each: font 8, 12, 24, 32 px, low contrast, corner placement. 7 settings x 7 text cells + 1 clean = 50.
- Strong defences at the default and at 24 px: blur sigma 1, 2, 3, JPEG 50, 20, 10, OCR-inpaint. 7 defences x (2 x 7 text cells + 1 clean) = 105.
- Once at the default: median, bit depth, down-up. 3 x 8 = 24.

In total 35,800 manifest rows and 16 GB of images, under Kaggle's 20 GB output limit.

**Why the images end up in zip files.** Kaggle keeps only about 500 files of a notebook's output, so 35,800 loose images cannot become a dataset. The last cell packs them into 10 zip files of 20 photos each and deletes the loose files. The output is then 12 files: the 10 zips, `full.csv` and `ocr_masks.zip`. The full-run notebooks take the images they need out of the zips.

**Before you start**
1. Settings: Accelerator **GPU T4 x1** (EasyOCR uses it), Internet **on**.
2. Add-ons > Secrets: `GITHUB_TOKEN` switched on.
3. Photo swaps from the pilot gate must already be committed to `data/attack_set.csv`. The notebook renders whatever that file says.

**How to run**
1. Quick test first: set `LIMIT = 2` in the settings cell and run all cells by hand. It should finish in a few minutes and print `OK`.
2. Then set `LIMIT = 0`, click **Save Version > Save & Run All (Commit)**. It runs in the background, about 1.5 to 2 hours.
3. When it finishes, open the version's **Output** tab. It should list `images_01.zip` to `images_10.zip`, `full.csv` and `ocr_masks.zip`, about 16 GB in all.
4. Make the output a dataset: on the Output tab use **New Dataset** (call it `pixelguard-full`), then **Share** it with Mufeed and Ali. They add it with Add Input in their full-run notebooks.

In [ ]:
!pip install -q easyocr
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, OCR will be very slow, turn on the GPU')

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.chdir('/kaggle/working')
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
LIMIT = 0
OUT = '/kaggle/working'
COCO = '/tmp/coco'

In [ ]:
import csv, sys
import pandas as pd
sys.path.insert(0, 'src')
import make_manifest as mm

DEFAULT = 'font16_high_object'
SWEEP = ['font8_high_object', 'font12_high_object', 'font24_high_object', 'font32_high_object',
         'font16_low_object', 'font16_high_corner']
STRONG = ['blur_s1', 'blur_s2', 'blur_s3', 'jpeg_q50', 'jpeg_q20', 'jpeg_q10', 'ocr_inpaint']
ONCE = ['median_5', 'bits_3', 'downup_05']

with open('data/attack_set.csv', encoding='utf-8') as f:
    photos = [r for r in csv.DictReader(f) if r['split'] in ('tune', 'test')]
photos = photos[:LIMIT] if LIMIT else photos
rows = (mm.build_manifest(photos, [DEFAULT] + SWEEP, ['none'])
        + mm.build_manifest(photos, [DEFAULT, 'font24_high_object'], STRONG)
        + mm.build_manifest(photos, [DEFAULT], ONCE))
manifest = pd.DataFrame(rows)
assert not manifest.duplicated(['image_id', 'cell', 'setting', 'defence']).any()
print(len(photos), 'photos,', len(manifest), 'manifest rows,', len(manifest) // max(1, len(photos)), 'per photo')
print(manifest.groupby('defence').size().to_string())

In [ ]:
import re, time

ids = ','.join(r['image_id'] for r in photos)
t0 = time.time()
for s in [DEFAULT] + SWEEP:
    size, contrast, placement = re.fullmatch(r'font(\d+)_([a-z]+)_([a-z]+)', s).groups()
    r = subprocess.run([sys.executable, 'src/render.py', '--ids', ids, '--font-size', size, '--contrast', contrast,
                        '--placement', placement, '--cache', COCO, '--out', f'{OUT}/images'],
                       capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1] if r.returncode == 0 else 'FAILED ' + s + '\n' + r.stderr[-2000:])
print(f'rendering took {(time.time() - t0) / 60:.0f} min')

In [ ]:
t0 = time.time()
todo = [r for r in rows if r['defence'] != 'none']
for i in range(0, len(todo), 2000):
    mm.generate_missing(todo[i:i + 2000], OUT)
    print(f'{min(i + 2000, len(todo))}/{len(todo)} defended rows done, {(time.time() - t0) / 60:.0f} min so far')

In [ ]:
import shutil, zipfile
from pathlib import Path

missing = [p for p in manifest['path'] if not (Path(OUT) / p).exists()]
masks = sorted(Path(OUT, 'images').glob('*_mask.png'))
with zipfile.ZipFile(f'{OUT}/ocr_masks.zip', 'w') as z:
    for m in masks:
        z.write(m, m.name)
        m.unlink()
manifest.to_csv(f'{OUT}/full.csv', index=False)
files = list(Path(OUT, 'images').glob('*.png'))
size_gb = sum(f.stat().st_size for f in files) / 1e9
print(len(manifest), 'manifest rows,', len(missing), 'missing files')
print(len(files), 'images,', f'{size_gb:.1f} GB,', len(masks), 'OCR masks moved into ocr_masks.zip')
print('OK' if not missing else 'PROBLEM: ' + ', '.join(missing[:5]))

In [ ]:
import math

SHARDS = 10
ids = [r['image_id'] for r in photos]
size = math.ceil(len(ids) / SHARDS)
packed = 0
for k in range(SHARDS):
    chunk = ids[k * size:(k + 1) * size]
    if not chunk:
        continue
    zip_path = Path(OUT, f'images_{k + 1:02d}.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
        for image_id in chunk:
            for f in sorted(Path(OUT, 'images').glob(f'{image_id}_*.png')):
                z.write(f, f'images/{f.name}')
                f.unlink()
                packed += 1
    print(zip_path.name, len(chunk), 'photos,', f'{zip_path.stat().st_size / 1e9:.2f} GB')
left = list(Path(OUT, 'images').glob('*'))
if not left:
    Path(OUT, 'images').rmdir()
print(packed, 'images packed,', len(left), 'left loose')
print('output files:', sorted(p.name for p in Path(OUT).iterdir()))
print('OK' if packed == len(manifest['path'].unique()) and not left else 'PROBLEM')